In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report


c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
random_state = 42

In [3]:
df = pd.read_csv('../Milestone 3 EDA/credit_card_cleaned.csv')
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


In [4]:
fraud = df[df['Class'] == 1]
non_fraud = df[df['Class'] == 0]
len(fraud), len(non_fraud)

(492, 284315)

In [5]:
# Undersample majority class to a 1:10 ratio
non_fraud_downsampled = non_fraud.sample(n=len(fraud) * 10, random_state=random_state)
non_fraud_downsampled

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
138028,82450.0,1.314539,0.590643,-0.666593,0.716564,0.301978,-1.125467,0.388881,-0.288390,-0.132137,...,-0.170307,-0.429655,-0.141341,-0.200195,0.639491,0.399476,-0.034321,0.031692,0.76,0
63099,50554.0,-0.798672,1.185093,0.904547,0.694584,0.219041,-0.319295,0.495236,0.139269,-0.760214,...,0.202287,0.578699,-0.092245,0.013723,-0.246466,-0.380057,-0.396030,-0.112901,4.18,0
73411,55125.0,-0.391128,-0.245540,1.122074,-1.308725,-0.639891,0.008678,-0.701304,-0.027315,-2.628854,...,-0.133485,0.117403,-0.191748,-0.488642,-0.309774,0.008100,0.163716,0.239582,15.00,0
164247,116572.0,-0.060302,1.065093,-0.987421,-0.029567,0.176376,-1.348539,0.775644,0.134843,-0.149734,...,0.355576,0.907570,-0.018454,-0.126269,-0.339923,-0.150285,-0.023634,0.042330,57.00,0
148999,90434.0,1.848433,0.373364,0.269272,3.866438,0.088062,0.970447,-0.721945,0.235983,0.683491,...,0.103563,0.620954,0.197077,0.692392,-0.206530,-0.021328,-0.019823,-0.042682,0.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249845,154606.0,-1.235823,-0.571536,1.620984,-2.648022,0.431772,1.315577,-0.258094,0.687327,-1.039031,...,0.163828,0.220473,-0.023553,-1.685719,0.301464,-0.260192,0.268870,0.102644,88.30,0
131485,79621.0,1.227929,0.065334,0.043523,0.109012,-0.394905,-1.307819,0.349677,-0.310790,-0.030846,...,-0.424623,-1.375408,0.178609,0.395001,0.078121,0.619247,-0.109945,0.009712,44.94,0
96599,65844.0,-0.878702,0.077973,0.345555,-1.646600,-0.090887,-0.930793,-0.136898,0.407723,-1.456907,...,0.367227,0.880295,-0.150938,0.039828,-0.197562,-0.330629,0.274606,0.090629,15.00,0
164556,116806.0,-1.296659,0.282590,1.651779,-1.503733,-1.575132,-0.079848,-0.673041,0.332639,0.226605,...,-0.141780,0.269977,-0.043272,-0.036735,-0.731235,0.545507,-0.173457,0.059645,45.51,0


In [6]:
# Combine and shuffle
df_reduced = pd.concat([fraud, non_fraud_downsampled]).sample(frac=1).reset_index(drop=True)
df_reduced

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,49916.0,1.326864,-0.618567,0.410831,-0.832726,-0.963446,-0.527423,-0.642853,-0.011497,-0.955348,...,-0.051417,-0.395356,0.059283,-0.025647,0.219387,-0.480360,-0.002653,0.013755,40.00,0
1,134026.0,1.834784,-1.510685,-0.907366,-0.879674,-0.834695,0.165743,-0.808196,0.089150,-0.073234,...,-0.094909,-0.645952,0.262278,0.316084,-0.508820,-0.619573,-0.032495,-0.022081,173.66,0
2,132476.0,2.071766,-0.716727,-0.700152,-0.308681,-1.456541,-1.665343,-0.936405,-0.167762,0.179696,...,0.285429,0.854991,0.227339,0.801181,-0.309154,-0.141515,0.022357,-0.004490,19.99,0
3,160900.0,1.325098,-3.072961,-1.031136,-1.203151,-2.037399,-0.185028,-0.906473,-0.177656,-0.971282,...,0.326376,0.291421,-0.189197,0.748730,-0.473857,-0.150235,-0.057461,0.053114,509.00,0
4,120696.0,0.779249,-2.097663,-2.564813,0.861715,-0.365678,-0.565834,0.871301,-0.298684,1.091925,...,0.120621,-0.957629,-0.369285,0.467858,-0.289945,-0.178880,-0.141730,0.095840,664.96,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5407,125140.0,0.730207,-3.062369,-3.421701,-0.401459,-0.443465,-0.966934,1.341823,-0.641162,-1.293683,...,0.996353,1.047648,-0.889626,0.883963,0.428929,0.162582,-0.259457,0.040409,804.40,0
5408,52222.0,-0.820159,1.533816,0.348268,0.959056,-0.803734,-0.505718,0.229739,0.679653,-1.253653,...,0.304265,0.649767,0.037323,0.576147,-0.134948,-0.370888,-0.273768,-0.104779,70.65,0
5409,40607.0,1.242017,-0.323744,0.918279,-0.720957,-1.158008,-0.779018,-0.549553,-0.083087,1.724524,...,0.148854,0.744351,-0.144613,0.459564,0.653366,-0.534556,0.100010,0.035381,4.32,0
5410,66630.0,-1.212812,1.078708,-0.658513,-2.242114,1.834271,3.650446,-1.514891,-1.264592,-0.646356,...,-1.281668,-0.009003,0.208746,1.021385,-0.531189,0.682470,-0.101852,0.094360,5.37,0


In [7]:
X = df_reduced.drop(columns='Class')
y = df_reduced['Class']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    stratify=y,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

In [9]:
cv = StratifiedKFold(n_splits=5)

### Recall as metric

In [10]:
# Parameter tuning
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    return scores.mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_recall_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_recall_df)

[I 2026-04-14 15:38:59,176] A new study created in memory with name: no-name-9b946a46-1eb8-4212-872d-5fdb85be3789
[I 2026-04-14 15:39:02,251] Trial 0 finished with value: 0.8375527426160337 and parameters: {'n_estimators': 200, 'max_depth': 32, 'min_samples_split': 7}. Best is trial 0 with value: 0.8375527426160337.
[I 2026-04-14 15:39:03,764] Trial 1 finished with value: 0.8299253489126907 and parameters: {'n_estimators': 161, 'max_depth': 2, 'min_samples_split': 4}. Best is trial 0 with value: 0.8375527426160337.
[I 2026-04-14 15:39:05,309] Trial 2 finished with value: 0.8299253489126907 and parameters: {'n_estimators': 176, 'max_depth': 2, 'min_samples_split': 2}. Best is trial 0 with value: 0.8375527426160337.
[I 2026-04-14 15:39:08,032] Trial 3 finished with value: 0.840084388185654 and parameters: {'n_estimators': 197, 'max_depth': 12, 'min_samples_split': 6}. Best is trial 3 with value: 0.840084388185654.
[I 2026-04-14 15:39:09,281] Trial 4 finished with value: 0.837552742616033

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
15,15,0.855339,2026-04-14 15:39:13.971138,2026-04-14 15:39:14.080308,0 days 00:00:00.109170,3,9,21,COMPLETE
44,44,0.855339,2026-04-14 15:39:21.508523,2026-04-14 15:39:21.601482,0 days 00:00:00.092959,3,10,19,COMPLETE
40,40,0.855339,2026-04-14 15:39:21.084272,2026-04-14 15:39:21.194061,0 days 00:00:00.109789,3,7,21,COMPLETE
41,41,0.855339,2026-04-14 15:39:21.194061,2026-04-14 15:39:21.287989,0 days 00:00:00.093928,3,7,20,COMPLETE
76,76,0.855339,2026-04-14 15:39:28.529172,2026-04-14 15:39:28.624588,0 days 00:00:00.095416,3,7,21,COMPLETE


In [11]:
# Train using optimal parameters
model_1 = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_recall_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_recall_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_recall_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model_1.fit(X_train, y_train)

,n_estimators,21
,criterion,'gini'
,max_depth,3
,min_samples_split,9
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
y_pred = model_1.predict(X_test)

In [13]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       985
           1       0.89      0.92      0.90        98

    accuracy                           0.98      1083
   macro avg       0.94      0.95      0.95      1083
weighted avg       0.98      0.98      0.98      1083



In [14]:
recall_model_report_df = classification_report(y_test, y_pred, output_dict=True)

## F1 as metric

In [15]:
# Parameter tuning
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    return scores.mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_f1_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_f1_df)

[I 2026-04-14 15:39:32,003] A new study created in memory with name: no-name-5c1d1bbc-e67a-4d6c-a63a-13c69b3dcad6
[I 2026-04-14 15:39:32,328] Trial 0 finished with value: 0.9075058004905301 and parameters: {'n_estimators': 36, 'max_depth': 27, 'min_samples_split': 6}. Best is trial 0 with value: 0.9075058004905301.
[I 2026-04-14 15:39:32,752] Trial 1 finished with value: 0.8906643026004726 and parameters: {'n_estimators': 152, 'max_depth': 2, 'min_samples_split': 5}. Best is trial 0 with value: 0.9075058004905301.
[I 2026-04-14 15:39:32,940] Trial 2 finished with value: 0.8738366363319672 and parameters: {'n_estimators': 56, 'max_depth': 2, 'min_samples_split': 8}. Best is trial 0 with value: 0.9075058004905301.
[I 2026-04-14 15:39:33,710] Trial 3 finished with value: 0.9053150594400272 and parameters: {'n_estimators': 111, 'max_depth': 7, 'min_samples_split': 2}. Best is trial 0 with value: 0.9075058004905301.
[I 2026-04-14 15:39:34,165] Trial 4 finished with value: 0.9000822345031019

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
62,62,0.912319,2026-04-14 15:40:19.280076,2026-04-14 15:40:20.260802,0 days 00:00:00.980726,9,9,121,COMPLETE
55,55,0.912319,2026-04-14 15:40:12.164743,2026-04-14 15:40:13.190240,0 days 00:00:01.025497,9,9,134,COMPLETE
56,56,0.912319,2026-04-14 15:40:13.190240,2026-04-14 15:40:14.296877,0 days 00:00:01.106637,9,9,134,COMPLETE
48,48,0.911093,2026-04-14 15:40:08.117321,2026-04-14 15:40:08.508671,0 days 00:00:00.391350,10,10,47,COMPLETE
44,44,0.910867,2026-04-14 15:40:04.785301,2026-04-14 15:40:05.854117,0 days 00:00:01.068816,9,9,130,COMPLETE


In [16]:
# Train using optimal parameters
model_2 = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_f1_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_f1_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_f1_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model_2.fit(X_train, y_train)

,n_estimators,121
,criterion,'gini'
,max_depth,9
,min_samples_split,9
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [17]:
y_pred = model_2.predict(X_test)

In [18]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       985
           1       0.96      0.91      0.93        98

    accuracy                           0.99      1083
   macro avg       0.97      0.95      0.96      1083
weighted avg       0.99      0.99      0.99      1083



In [19]:
f1_model_report_df = classification_report(y_test, y_pred, output_dict=True)

### Analyze reports

In [20]:
# 2. Convert and transpose
recall_report_df = pd.DataFrame(recall_model_report_df).transpose()
f1_report_df = pd.DataFrame(f1_model_report_df).transpose()

# 3. Combine multiple reports (e.g., adding a 'Model' column for identification)
combined_df = pd.concat([recall_report_df, f1_report_df], keys=['RF optimized for Recall', 'RF optimized for F1'])

In [21]:
combined_df

precision    recall  f1-score  \
RF optimized for Recall 0              0.991853  0.988832  0.990341   
                        1              0.891089  0.918367  0.904523   
                        accuracy       0.982456  0.982456  0.982456   
                        macro avg      0.941471  0.953600  0.947432   
                        weighted avg   0.982735  0.982456  0.982575   
RF optimized for F1     0              0.990909  0.995939  0.993418   
                        1              0.956989  0.908163  0.931937   
                        accuracy       0.987996  0.987996  0.987996   
                        macro avg      0.973949  0.952051  0.962677   
                        weighted avg   0.987840  0.987996  0.987854   

                                          support  
RF optimized for Recall 0              985.000000  
                        1               98.000000  
                        accuracy         0.982456  
                        macro avg     1083.000000  
                        weighted avg  1083.000000  
RF optimized for F1     0              985.000000  
                        1               98.000000  
                        accuracy         0.987996  
                        macro avg     1083.000000  
                        weighted avg  1083.000000

In [23]:
# Recall model
importances = model_1.feature_importances_
feature_names = df.drop(columns='Class').columns
feature_imp_recall_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)
feature_imp_recall_df

,Feature,Gini Importance
14,V14,0.385421
17,V17,0.166683
12,V12,0.085889
10,V10,0.073857
4,V4,0.064483
3,V3,0.043349
7,V7,0.041566
16,V16,0.038438
2,V2,0.028712
9,V9,0.022989


In [24]:
# Recall model
importances = model_2.feature_importances_
feature_names = df.drop(columns='Class').columns
feature_imp_f1_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)
feature_imp_f1_df

,Feature,Gini Importance
14,V14,0.175584
10,V10,0.125815
4,V4,0.120538
12,V12,0.104941
17,V17,0.094660
16,V16,0.049719
11,V11,0.048258
3,V3,0.045854
2,V2,0.033576
7,V7,0.029487
